# MedDial on Colab: dialogues, validation, evaluation, tables

The third stage. `meddial-cohort` selected the admissions, `meddial-scr` extracted a Structured
Clinical Reference for each. This notebook takes those references and runs the rest:

1. build the case file the runner reads, from the references on disk;
2. **validate** it before spending hours — every selected case present, every reference parseable,
   evidence actually attached, and the judge drawn from a different family than the generator;
3. a two-case smoke run, so a broken setting fails in minutes rather than overnight;
4. `meddial-run` for each architecture variant, which generates the dialogues **and scores them** —
   structural validity, naturalness, patient and doctor factuality, knowledge boundary;
5. `meddial-tables`, which regenerates every manuscript table and the primary figure from the
   attempt records, with bootstrap confidence intervals.

**Evaluation is not a separate step.** The runner scores each dialogue as it produces it and retries
against the thresholds in the config, so `attempts.jsonl` already carries every score. `meddial-tables`
aggregates; it does not re-judge.

**What leaves this runtime: nothing.** Both models are served inside the runtime over loopback. The
references are MIMIC-derived, so everything here is restricted and dies with the runtime.

## 0. The card, and the two models that have to share it

In [ ]:
!nvidia-smi

This stage runs **two** models at once: a generator that plays the patient and the doctor, and an
evaluator that reads each transcript into claims and then rules on them. Both stay resident, so the
card has to hold the pair — a constraint that did not exist during extraction, where one model ran
alone.

In [ ]:
import os, subprocess, time
import httpx

# Same reasoning as the extraction notebook: Ollama's 4,096-token default silently
# truncates from the front, and a dialogue prompt carries the reference plus the
# transcript so far, which grows with every turn.
try:
    import torch
    _gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
except ImportError:
    _gib = 0

CONTEXT = 32768 if _gib >= 38 else 16384
os.environ['OLLAMA_CONTEXT_LENGTH'] = str(CONTEXT)

subprocess.Popen(
    ['ollama', 'serve'],
    env={**os.environ},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(60):
    try:
        httpx.get('http://localhost:11434/api/tags', timeout=2.0).raise_for_status()
        print('ollama is serving')
        break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError('ollama did not come up; re-run this cell')

print(f'{_gib:.0f} GiB of VRAM -> {CONTEXT}-token window')

### Three roles, three families, and the family the reference already used

`meddial-run` warns rather than refuses, but a judge that shares a family with the model it is
checking inherits that model's blind spots and the score stops being a check. Two independence
constraints apply here, not one:

* the judge must differ from the **generator**, or it grades its own lineage's dialogue;
* the judge should differ from whatever extracted the **reference**, because faithfulness and
  knowledge-boundary are scored *against* that reference. If you extracted with `qwen3.5:35b`, a
  Qwen judge cannot see what a Qwen extractor missed.

So the CLI default judge (`qwen3.5:9b`) is the wrong choice after a Qwen extraction, and this
notebook does not use it. The assignment below is three distinct lineages, none of them Qwen.

Claim extraction gets its own model rather than borrowing the judge. Implementation Plan A.2 puts it
on `gpt-oss:20b`: it is the highest-volume path in the evaluator — every transcript, every attempt —
and what it needs is JSON reliability, not world knowledge.

Faithfulness is **two** model calls, not one: a claim extractor decides *what was claimed*, and the
judge decides *whether each claim is true*. The score is computed over the claims the first stage
produced, so a claim that is never extracted can never be marked unfaithful — it is absent from the
denominator, and a missed fabrication **raises** the score rather than lowering it.

That makes claim recall the ceiling on faithfulness, and recall is a matter of how capable the
extracting model is. A small extractor feeding a large judge caps the metric at the small model's
recall, however good the judge is. So both stages run on **one large model** here rather than being
split across a small extractor and a separate judge: `--claim-extractor` defaults to the judge, and
this notebook leaves it there deliberately.

What that gives up is minor and worth naming: a shared model cannot phrase a claim in one stage and
be surprised by it in the next, so mutually-agreeable phrasing of *extracted* claims is not checked.
That is second-order — the judge never sees an unextracted claim either way — and it is what
`ensemble.py` (EVAL-8, ≥3 judge families with agreement) is for once it exists.

The independence that does matter is preserved on both axes: the evaluator differs from the
generator, and from the family that extracted the references.

| Role | Model | Lineage | ~Q4 weights |
| --- | --- | --- | --- |
| reference (already done) | `qwen3.5:35b` | Qwen | — |
| generator (patient **and** doctor) | `gemma4:31b` | Google | 18.5 GiB |
| judge **and** claim extraction | `llama3.3:70b` | Meta | 39.6 GiB |

58.1 GiB of weights (sizes read from the Ollama manifests, not estimated) plus two KV caches at a
32,768-token window — around 70 GiB, which is what an 80 GiB card is for, without much to spare.
Below 70 GiB the cell falls back to `gpt-oss:20b`, and the residency check two cells down reports
the actual split before anything long starts.

One model plays **both** sides of the dialogue. That is a deliberate choice and it has a cost
worth stating in the write-up: patient and doctor share priors, so the doctor tends to ask what
the patient is disposed to answer. It plausibly flatters naturalness and makes a disclosure-policy
failure harder to see. `meddial-run` takes one `--generator` for both roles, and the configs now
record `gemma4:31b` under both so the manifest names the model that actually spoke rather than two
that did not.

In [ ]:
# The family that extracted the references. Everything else is chosen to avoid
# it, because the reference is what faithfulness and knowledge-boundary are
# scored against.
SCR_EXTRACTOR_FAMILY = 'qwen'

# One model plays the patient and the doctor. meddial-run wires a single
# generator provider for both, and the configs now record gemma4:31b for both
# roles so the run manifest names the model that actually spoke.
GENERATOR, GENERATOR_FAMILY = 'gemma4:31b', 'gemma'

# One model judges and extracts claims. Claim recall is the ceiling on
# faithfulness -- an unextracted claim is never scored -- so the capability of
# the extracting model matters more than holding it in a separate family from
# the judge, which only guards against correlated phrasing of the claims that
# were extracted anyway.
#
# Real download sizes from the Ollama manifests: gemma4:31b is 18.5 GiB and
# llama3.3:70b is 39.6 GiB, so 58.1 GiB of weights plus a KV cache each at the
# window set above -- around 70 GiB. That is what 80 GiB buys, with little to
# spare; below it, step down to the smaller evaluator.
if _gib >= 70:
    JUDGE, JUDGE_FAMILY = 'llama3.3:70b', 'llama'      # A100/H100 80GB
elif _gib >= 30:
    JUDGE, JUDGE_FAMILY = 'gpt-oss:20b', 'gpt-oss'     # A100 40GB
else:
    JUDGE, JUDGE_FAMILY = 'gpt-oss:20b', 'gpt-oss'     # smaller cards will swap

families = {'generator': GENERATOR_FAMILY, 'judge + claims': JUDGE_FAMILY}
assert families['judge + claims'] != families['generator'], (
    'the evaluator shares the generator family and would grade its own lineage'
)
assert families['judge + claims'] != SCR_EXTRACTOR_FAMILY, (
    f'the evaluator is {JUDGE_FAMILY!r}, the family that extracted the references -- it cannot '
    'see what its own lineage missed'
)

os.environ['MEDDIAL_GENERATOR'] = GENERATOR
os.environ['MEDDIAL_JUDGE'] = JUDGE
os.environ['MEDDIAL_JUDGE_FAMILY'] = JUDGE_FAMILY
os.environ['MEDDIAL_GENERATOR_FAMILY'] = GENERATOR_FAMILY

for role, model in (('generator', GENERATOR), ('judge + claims', JUDGE)):
    print(f'{role:16} {model:24} ({families[role]})')
print(f'\nreferences were extracted by a {SCR_EXTRACTOR_FAMILY!r} model; neither role uses it')

Tags are checked against the registry before anything downloads — the Implementation Plan names
`qwen3.5:32b`, which Ollama does not publish, so a plausible-looking tag is not evidence.

In [ ]:
# Ask for the manifest: registry.ollama.ai does not implement the Docker-style
# tags/list endpoint, so that path 404s even for models that plainly exist.
def is_published(ref):
    family, _, tag = ref.partition(':')
    url = f'https://registry.ollama.ai/v2/library/{family}/manifests/{tag or "latest"}'
    try:
        return httpx.get(url, timeout=30.0).status_code == 200
    except Exception as exc:
        print(f'Could not reach the Ollama registry ({type(exc).__name__}: {exc}).')
        return None


for ref in (GENERATOR, JUDGE):
    published = is_published(ref)
    if published is False:
        raise SystemExit(
            f'{ref!r} is not published by Ollama. Browse '
            f'https://ollama.com/library/{ref.partition(":")[0]}/tags and edit the cell above.'
        )
    print(f'{ref}: {"published" if published else "unverified"}')

In [ ]:
!ollama pull $MEDDIAL_GENERATOR
!ollama pull $MEDDIAL_JUDGE

### Confirm both fit on the GPU before starting

`PROCESSOR` must read 100% GPU for **both**. If either is partly on CPU the run does not fail, it
crawls — and a dialogue run is many times more model calls than extraction was, because every turn
is a call and every attempt is scored.

In [ ]:
for model in (GENERATOR, JUDGE):
    httpx.post(
        'http://localhost:11434/api/generate',
        json={'model': model, 'prompt': 'hi', 'options': {'num_predict': 1}},
        timeout=900.0,
    ).raise_for_status()

resident = httpx.get('http://localhost:11434/api/ps', timeout=60.0).json().get('models', [])
total = 0
for entry in resident:
    size, vram = entry.get('size') or 0, entry.get('size_vram') or 0
    on_gpu = vram / size if size else 0.0
    total += size
    print(f"{entry.get('name')}: context {entry.get('context_length')}, "
          f"{on_gpu:.0%} of {size / 1024**3:.1f} GiB on GPU")
    if on_gpu < 0.999:
        print(f"  !! only {on_gpu:.0%} on GPU -- this pair is too big for this card.")
print(f'\nboth resident: {total / 1024**3:.1f} GiB of {_gib:.0f} GiB')

## 1. Install MedDial

The thin install is enough. `meddial-run` and `meddial-tables` import none of the `eval` extra --
no DeepEval, no transformers, no torch beyond what Colab already ships. The judging here is done by
the local Qwen model over HTTP, not by a scoring library.

In [ ]:
import os, subprocess

REPO = '/content/FinalProject-MedDial'
URL = 'https://github.com/alongott15/FinalProject-MedDial.git'
BRANCH = 'main'


def git(*args: str) -> str:
    done = subprocess.run(['git', '-C', REPO, *args], capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr.strip()}")
    return done.stdout.strip()


if os.path.isdir(f'{REPO}/.git'):
    local = git('status', '--porcelain')
    if local:
        print('Discarding local changes in the clone:')
        print(local)
    git('fetch', '--quiet', 'origin', BRANCH)
    git('reset', '--hard', '--quiet', f'origin/{BRANCH}')
    git('clean', '-qfd')
else:
    done = subprocess.run(
        ['git', 'clone', '--quiet', '--branch', BRANCH, URL, REPO],
        capture_output=True, text=True,
    )
    if done.returncode:
        raise RuntimeError(f'git clone failed:\n{done.stderr.strip()}')

print('now at', git('log', '-1', '--pretty=%h %s'))

In [ ]:
%pip install -q -e '/content/FinalProject-MedDial'

## 2. Bring the references in

This notebook starts where `meddial_colab.ipynb` stopped: a directory of `scr_<subject>_<hadm>.md`
files and the cohort manifest they were selected from. A Colab runtime does not survive between
sessions, so unless you are continuing in the same one, upload the archive that notebook packaged
and unpack it here.

Both paths below are what the extraction notebook wrote. Edit them if you put them elsewhere.

In [ ]:
from pathlib import Path

REFERENCES = Path('/content/meddial-out/references')
COHORT_MANIFEST = Path('/content/meddial-out/cohort/cohort_private_manifest.json')

# If you are resuming in a fresh runtime, upload meddial-out.zip first and unpack it:
#   from google.colab import files; files.upload()
#   !unzip -q -o meddial-out.zip -d /content

if not REFERENCES.is_dir():
    raise SystemExit(
        f'No references at {REFERENCES}. Run notebooks/meddial_colab.ipynb first, or upload and '
        'unpack the archive it produced (see the commented lines above).'
    )

found = sorted(REFERENCES.glob('scr_*.md'))
print(f'{len(found)} reference file(s) in {REFERENCES}')

## 3. Build the case file

`meddial-run` reads JSONL, one case per line, each carrying the reference the architecture works
from. That is assembled here from the references on disk rather than kept as a separate artefact,
so the cases cannot drift from the references they claim to describe.

`note_id` is set to the case id because that is what `meddial-scr` stamped into every evidence span;
they have to agree or the spans resolve against a note nobody supplied.

In [ ]:
import json
from Utils.markdown_gtmf import load_gtmf_markdown
from meddial.knowledge.reference import StructuredClinicalReference

N_CASES = 10          # start small; None for every reference
CASES = Path('/content/meddial-out/cases.jsonl')

rows, unreadable = [], []
for path in found:
    try:
        payload = load_gtmf_markdown(str(path))
        reference = StructuredClinicalReference.model_validate(payload)
    except Exception as exc:
        unreadable.append((path.name, f'{type(exc).__name__}: {exc}'))
        continue
    rows.append({
        'case_id': reference.case_id,
        'note_id': reference.case_id,
        'reference': payload,
        '_reference_model': reference,      # kept for the checks below, stripped before writing
    })

rows.sort(key=lambda row: row['case_id'])
selected = rows if N_CASES is None else rows[:N_CASES]
print(f'{len(rows)} reference(s) parsed, {len(unreadable)} unreadable, {len(selected)} in this run')
for name, why in unreadable[:10]:
    print(f'  {name}: {why}')

## 4. Validate before spending the GPU on it

Four checks, in the order that a failure is cheapest to fix. None of them calls a model.

The third is the one worth reading. A reference whose entities carry no evidence is not a reference:
faithfulness and knowledge-boundary scoring are measured against it, so an ungrounded entity is
scored against nothing. It does not stop the run — extraction recall is itself a measured quantity
(GRND-1/2) — but you should know the number before it becomes a result.

In [ ]:
problems = []

# 1. Does every case the cohort selected have a reference?
if COHORT_MANIFEST.is_file():
    manifest = json.loads(COHORT_MANIFEST.read_text(encoding='utf-8'))
    wanted = {f"{c['subject_id']}_{c['hadm_id']}" for c in manifest.get('selected', [])}
    have = {row['case_id'] for row in rows}
    missing = wanted - have
    print(f'cohort selected {len(wanted)}, references present for {len(wanted & have)}')
    if missing:
        print(f'  {len(missing)} selected case(s) have no reference -- extraction failed on these.')
        print(f'  first few: {sorted(missing)[:5]}')
        problems.append('references missing for part of the cohort')
else:
    print(f'no cohort manifest at {COHORT_MANIFEST}; skipping the coverage check')

# 2. Anything that would not parse.
if unreadable:
    problems.append(f'{len(unreadable)} reference file(s) unreadable')

# 3. Grounding, and empty cores.
empty, ungrounded, entity_total, unevidenced_total = [], [], 0, 0
for row in selected:
    reference = row['_reference_model']
    core = reference.core
    if not (core.symptoms or core.diagnoses or core.treatments):
        empty.append(row['case_id'])
    missing_evidence = reference.unevidenced_entities()
    entities = (len(core.symptoms) + len(core.diagnoses) + len(core.treatments)
                + len(reference.context.current_medications)
                + len(reference.context.discharge_medications))
    entity_total += entities
    unevidenced_total += len(missing_evidence)
    if missing_evidence:
        ungrounded.append((row['case_id'], len(missing_evidence), entities))

print(f'\nentities across this run: {entity_total}, of which {unevidenced_total} carry no evidence '
      f'({unevidenced_total / entity_total:.0%})' if entity_total else '\nno entities at all')
for case_id, missing_count, entities in sorted(ungrounded, key=lambda r: -r[1])[:5]:
    print(f'  {case_id}: {missing_count}/{entities} unevidenced')
if empty:
    print(f'\n{len(empty)} reference(s) have no symptoms, diagnoses or treatments: {empty[:5]}')
    problems.append(f'{len(empty)} empty reference(s)')

# 4. Judge independence.
if GENERATOR_FAMILY == JUDGE_FAMILY:
    problems.append('judge shares the generator family')

print()
print('PROBLEMS:' if problems else 'No blocking problems.')
for problem in problems:
    print(f'  - {problem}')

Write the file only once the checks have been read. The helper model is stripped here — a case line
carries the reference payload, nothing else.

In [ ]:
CASES.parent.mkdir(parents=True, exist_ok=True)
with CASES.open('w', encoding='utf-8') as handle:
    for row in selected:
        handle.write(json.dumps({k: v for k, v in row.items() if not k.startswith('_')},
                                ensure_ascii=False) + '\n')

print(f'wrote {len(selected)} case(s) to {CASES}')
print(f'{CASES.stat().st_size / 1024:.0f} KiB')

### Optional: the `direct_llm` arm needs the note itself

Four of the five variants work from the reference. `direct_llm` exists to show what happens
*without* one, so it needs the raw note — which lives in MIMIC, not in the reference. Run this only
if you intend to include that arm, and only with the BigQuery cache the cohort notebook already
downloaded, so it costs nothing extra.

In [ ]:
ATTACH_NOTE_TEXT = False     # True to make the direct_llm arm runnable

if ATTACH_NOTE_TEXT:
    from meddial.cohort.mimic_bigquery import MimicBigQuerySource

    source = MimicBigQuerySource(
        os.environ['MIMIC_BIGQUERY_PROJECT'], cache_dir='/content/mimic-cache'
    )
    wanted_ids = {row['case_id'] for row in selected}
    notes = {
        f'{record.subject_id}_{record.hadm_id}': record.note_text
        for record in source.admission_records()
        if f'{record.subject_id}_{record.hadm_id}' in wanted_ids
    }
    with CASES.open('w', encoding='utf-8') as handle:
        for row in selected:
            case = {k: v for k, v in row.items() if not k.startswith('_')}
            case['note_text'] = notes.get(row['case_id'], '')
            handle.write(json.dumps(case, ensure_ascii=False) + '\n')
    print(f'attached note text to {sum(1 for r in selected if notes.get(r["case_id"]))} case(s)')
    print('this file now contains raw MIMIC notes -- it is restricted, like everything else here')
else:
    print('skipped; direct_llm will refuse these cases, the other four variants will not')

## 5. Smoke run: two cases, one variant

A dialogue run is many model calls per case — every turn, plus scoring, plus a retry whenever a
dimension misses its threshold. Getting that wrong at case 1 of 200 costs a night. Two cases first.

In [ ]:
SMOKE = Path('/content/meddial-out/cases-smoke.jsonl')
SMOKE.write_text(
    ''.join(line for line in CASES.read_text(encoding='utf-8').splitlines(keepends=True)[:2]),
    encoding='utf-8',
)
os.environ['SMOKE_PATH'] = str(SMOKE)
print(f'{SMOKE}: 2 case(s)')

In [ ]:
!meddial-run \
    --config /content/FinalProject-MedDial/configs/experiments/full_meddial.json \
    --cases $SMOKE_PATH \
    --out /content/meddial-out/runs-smoke \
    --generator $MEDDIAL_GENERATOR \
    --generator-family $MEDDIAL_GENERATOR_FAMILY \
    --judge $MEDDIAL_JUDGE \
    --judge-family $MEDDIAL_JUDGE_FAMILY

## 6. The run

One cell per architecture, because that is what the comparison is: the same cases and the same
policy through five pipelines. `meddial-run` generates each dialogue, scores it on all five
dimensions, and retries against the thresholds in the config, writing one immutable attempt record
per try.

It is resumable. Pass `--run-id` with the id a previous run printed to continue rather than start
over.

Run every variant for the full comparison, or cut the list to the arms you need. `direct_llm` is
excluded unless you attached note text above.

In [ ]:
VARIANTS = [
    'full_meddial',
    'knowledge_controlled',
    'structured_single_agent',
    'basic_multi_agent',
]
if ATTACH_NOTE_TEXT:
    VARIANTS.append('direct_llm')

print('will run:', ', '.join(VARIANTS))

In [ ]:
import subprocess

RUNS = Path('/content/meddial-out/runs')

for variant in VARIANTS:
    config = f'/content/FinalProject-MedDial/configs/experiments/{variant}.json'
    print(f'\n{"=" * 70}\n{variant}\n{"=" * 70}')
    done = subprocess.run(
        ['meddial-run',
         '--config', config,
         '--cases', str(CASES),
         '--out', str(RUNS),
         '--generator', GENERATOR, '--generator-family', GENERATOR_FAMILY,
         '--judge', JUDGE, '--judge-family', JUDGE_FAMILY],
        text=True,
    )
    if done.returncode:
        print(f'!! {variant} exited {done.returncode}; the variants after it still run')

## 7. Evaluation: the tables

`meddial-tables` pools the attempt records and regenerates every manuscript table and the primary
figure, with bootstrap confidence intervals over 2,000 resamples clustered by case. It does not
re-judge anything: the scores were written when the dialogues were.

M4 requires that no number in the write-up is transcribed by hand, which is why this is one command
over the records rather than a spreadsheet.

In [ ]:
attempts = sorted(RUNS.rglob('attempts.jsonl'))
if not attempts:
    raise SystemExit(f'no attempts.jsonl under {RUNS}; the runs above produced nothing')

for path in attempts:
    print(f'  {path}  ({sum(1 for _ in path.open()) } record(s))')
os.environ['MEDDIAL_ATTEMPTS'] = ' '.join(str(p) for p in attempts)

In [ ]:
!meddial-tables \
    --attempts $MEDDIAL_ATTEMPTS \
    --out /content/meddial-out/analysis \
    --run-id colab-pass-1

In [ ]:
ANALYSIS = Path('/content/meddial-out/analysis')
for path in sorted(ANALYSIS.rglob('*')):
    if path.is_file():
        print(f'{path.relative_to(ANALYSIS)}  ({path.stat().st_size / 1024:.1f} KiB)')

### The acceptance rate, per variant

In [ ]:
from collections import Counter

by_variant = {}
for path in attempts:
    for line in path.open(encoding='utf-8'):
        record = json.loads(line)
        variant = record.get('variant', '?')
        acceptance = record.get('evaluation', {}).get('acceptance', {})
        overall = acceptance.get('overall') if isinstance(acceptance, dict) else None
        accepted = str(overall).upper().endswith('ACCEPT')
        by_variant.setdefault(variant, Counter())[accepted] += 1

for variant, counts in sorted(by_variant.items()):
    total = sum(counts.values())
    print(f'{variant:26} {counts[True]:4}/{total:<4} accepted  ({counts[True] / total:.0%})')

## 8. Take the output with you

Everything under `/content/meddial-out` is derived from restricted data and disappears with the
runtime. The analysis directory is the part you will actually write from; the attempt records are
what makes it reproducible, so keep both.

In [ ]:
!cd /content && zip -qr meddial-run.zip meddial-out/runs meddial-out/analysis && ls -lh /content/meddial-run.zip

# from google.colab import files; files.download('/content/meddial-run.zip')